In [1]:
#Importing Libraries

from collections import Counter
import json
from pathlib import Path
import random
import pandas as pd
import spacy

In [2]:
# Confirming and setting project path

BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "Data" / "Raw" #Contains datasets csv
PROJECT_ROOT = BASE_DIR / "Data" / "Processed"
SPLIT_DATA = BASE_DIR / "Data" / "Splits"
SRC = BASE_DIR / "Src" / "Training"
MODEL_PATH = BASE_DIR / "Models" / "spacy_baseline"
MODEL_PATH_BEST = BASE_DIR / "Models" / "spacy_baseline" / "model-best"
print(BASE_DIR)
print(RAW_DIR)
print(PROJECT_ROOT)
print(SPLIT_DATA)
print(SRC)
print(MODEL_PATH)
print(MODEL_PATH_BEST)

D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Raw
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Processed
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Data\Splits
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Src\Training
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Models\spacy_baseline
D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Final\Models\spacy_baseline\model-best


In [3]:
nlp = spacy.load(MODEL_PATH_BEST)

print("Model loaded successfully")

Model loaded successfully


In [4]:
# Quick Inference Test

text = """
Patient reports severe headache and fever.
Diagnosed with diabetes.
Started Paracetamol 500 mg yesterday.
Scheduled MRI scan next week.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

severe headache -> SYMPTOM
fever -> SYMPTOM
diabetes -> SYMPTOM
Paracetamol -> MEDICATION
500 mg yesterday -> DOSAGE
Scheduled MRI scan -> DISEASE
next week -> SYMPTOM


In [5]:
# Reusable Prediction Function

def extract_entities(text):
    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

    return pd.DataFrame(entities)

In [6]:
# Test
extract_entities(
    "Patient has diabetes and takes Metformin 500 mg daily."
)

,text,label,start,end
0,diabetes,SYMPTOM,12,20
1,Metformin,MEDICATION,31,40
2,500 mg daily,DOSAGE,41,53


In [7]:
from spacy.tokens import DocBin

train_docs = list(
    DocBin().from_disk("../Data/Processed/train.spacy")
           .get_docs(nlp.vocab)
)

valid_docs = list(
    DocBin().from_disk("../Data/Processed/valid.spacy")
           .get_docs(nlp.vocab)
)

test_docs = list(
    DocBin().from_disk("../Data/Processed/test.spacy")
           .get_docs(nlp.vocab)
)

print("Train:", len(train_docs))
print("Valid:", len(valid_docs))
print("Test :", len(test_docs))

Train: 4000
Valid: 500
Test : 500


In [8]:
print(len(test_docs))

from pathlib import Path

test_path = "../Data/Processed/test.spacy"
print(Path(test_path).exists())

500
True


In [9]:
# Load test.spacy

from spacy.tokens import DocBin

test_path = r"D:\PycharmProjects\DS Projects\Healthcare NLP Auto-Labeling System_Prac\Data\Processed\test.spacy"

doc_bin = DocBin().from_disk(test_path)

test_docs = list(doc_bin.get_docs(nlp.vocab))

print("Test documents:", len(test_docs))

Test documents: 500


In [10]:
# Evaluation

from spacy.scorer import Scorer
from spacy.training import Example

scorer = Scorer()

examples = []

for gold_doc in test_docs:

    pred_doc = nlp(gold_doc.text)

    examples.append(
        Example(pred_doc, gold_doc)
    )

scores = scorer.score(examples)
scores

{'token_acc': 1.0,
 'token_p': 1.0,
 'token_r': 1.0,
 'token_f': 1.0,
 'sents_p': None,
 'sents_r': None,
 'sents_f': None,
 'tag_acc': None,
 'pos_acc': None,
 'morph_acc': None,
 'morph_micro_p': None,
 'morph_micro_r': None,
 'morph_micro_f': None,
 'morph_per_feat': None,
 'dep_uas': None,
 'dep_las': None,
 'dep_las_per_type': None,
 'ents_p': 1.0,
 'ents_r': 1.0,
 'ents_f': 1.0,
 'ents_per_type': {'DISEASE': {'p': 1.0, 'r': 1.0, 'f': 1.0},
  'MEDICATION': {'p': 1.0, 'r': 1.0, 'f': 1.0},
  'DOSAGE': {'p': 1.0, 'r': 1.0, 'f': 1.0},
  'SYMPTOM': {'p': 1.0, 'r': 1.0, 'f': 1.0},
  'PROCEDURE': {'p': 1.0, 'r': 1.0, 'f': 1.0}},
 'cats_score': 0.0,
 'cats_score_desc': 'macro F',
 'cats_micro_p': 0.0,
 'cats_micro_r': 0.0,
 'cats_micro_f': 0.0,
 'cats_macro_p': 0.0,
 'cats_macro_r': 0.0,
 'cats_macro_f': 0.0,
 'cats_macro_auc': 0.0,
 'cats_f_per_type': {},
 'cats_auc_per_type': {}}

In [11]:
# Pretty Metrics

print("Precision :", round(scores["ents_p"], 4))
print("Recall    :", round(scores["ents_r"], 4))
print("F1 Score  :", round(scores["ents_f"], 4))

Precision : 1.0
Recall    : 1.0
F1 Score  : 1.0


In [12]:
# Per Entity Performance

scores["ents_per_type"]

{'DISEASE': {'p': 1.0, 'r': 1.0, 'f': 1.0},
 'MEDICATION': {'p': 1.0, 'r': 1.0, 'f': 1.0},
 'DOSAGE': {'p': 1.0, 'r': 1.0, 'f': 1.0},
 'SYMPTOM': {'p': 1.0, 'r': 1.0, 'f': 1.0},
 'PROCEDURE': {'p': 1.0, 'r': 1.0, 'f': 1.0}}

In [13]:
# Error Analysis

errors = []

for gold_doc in test_docs:

    pred_doc = nlp(gold_doc.text)

    gold_entities = {
        (ent.start_char, ent.end_char, ent.label_)
        for ent in gold_doc.ents
    }

    pred_entities = {
        (ent.start_char, ent.end_char, ent.label_)
        for ent in pred_doc.ents
    }

    missed = gold_entities - pred_entities
    extra = pred_entities - gold_entities

    if missed or extra:
        errors.append({
            "text": gold_doc.text,
            "missed": missed,
            "extra": extra
        })

print("Documents with errors:", len(errors))

Documents with errors: 0









### Limitation
The model was trained on a synthetic template-based dataset.

Evaluation scores are expected to be very high because training, validation, and test data originate from the same distribution.

The model may not generalize well to unseen clinical notes, especially when entity capitalization or phrasing differs from the generated templates.

In [14]:
# Test more entities

extract_entities(
    "Patient diagnosed with hypertension and prescribed Metformin 500 mg."
)

,text,label,start,end
0,hypertension,DISEASE,23,35
1,Metformin,MEDICATION,51,60
2,500 mg.,DOSAGE,61,68


In [15]:
extract_entities(
    "Patient diagnosed with Pneumonia and prescribed Amoxicillin 250 mg."
)

,text,label,start,end
0,Pneumonia,DISEASE,23,32
1,Amoxicillin,MEDICATION,48,59
2,250 mg.,DOSAGE,60,67


In [16]:
doc = nlp("Patient has diabetes mellitus and takes Metformin 500 mg daily.")

for ent in doc.ents:
    print(
        repr(ent.text),
        ent.label_,
        ent.start_char,
        ent.end_char
    )

'diabetes mellitus' DISEASE 12 29
'Metformin' MEDICATION 40 49
'500 mg daily' DOSAGE 50 62


In [17]:
# Quick Inference Test

text = """
Patient has HTN. Started on lovastatin 20 mg bid.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

HTN -> DISEASE
lovastatin -> MEDICATION
20 mg bid -> DOSAGE


In [18]:
# Quick Inference Test

text = """
Patient recently diagnosed with HTN. Started on lovastatin 20 mg bid.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

HTN -> DISEASE
lovastatin -> MEDICATION
20 mg bid -> DOSAGE


In [19]:
# Quick Inference Test

text = """
Patient with DM. Started on lisinopril 20 mg bid
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

DM -> DISEASE
lisinopril -> MEDICATION
20 mg bid -> DOSAGE


In [20]:
# Quick Inference Test

text = """
Patient had a blood pressure reading of 120/180 mmHg. Started on lovastatin 20 mg bid.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

blood pressure reading -> SYMPTOM
120/180 mmHg -> DISEASE
lovastatin -> MEDICATION
20 mg bid -> DOSAGE


In [21]:
text = """
Patient has HTN and was prescribed Metformin 500 mg.
"""

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

HTN -> DISEASE
Metformin -> MEDICATION
500 mg. -> DOSAGE


### Disease abbreviation augmentation improved recognition of
### Common clinical abbreviations such as DM and HTN.

Before retraining: DM -> SYMPTOM HTN -> SYMPTOM

After retraining: DM -> DISEASE HTN -> DISEASE